## Employment-Trend Data Prep

By Ken Burchfiel

Released under the MIT License

This notebook is used to calculate the values on which my employment-trends HTML dashboard is based.

In [1]:
# import gc
# gc.collect()

In [2]:
import time
start_time = time.time()
from datetime import datetime
import pandas as pd
pd.set_option('display.max_columns', 1000)
import plotly.express as px
import numpy as np
import svy # prerequisite: pip install svy within your Python environment
# The svy library allows us to analyze complex survey
# data. See https://svylab.com/docs/svy/.
import polars as pl
import gc
import os
from survey_analysis_functions import perform_svy_analysis, \
create_svy_reg, survey_pivot

Configuration variables:

In [3]:
run_svy_analyses = False # svy-based point-estimate and confidence-interval 
# calculations will only get run when this filter is set to True.
run_pandas_analyses = False # an equivalent variable for Pandas-based
# pivot-table analyses
#filename_substrings_to_redo = []
filename_substrings_to_redo = ['2026']
# When a filename containing 
# svy results is present within this list, a new version of that analysis
# will get rerun. (Normally, analyses are skipped when a corresponding
# result file is already present.) This variable will prove particularly 
# useful for updating data for the current year.

# Specifying folders that will store svy- and Pandas-based analyses:
svy_period_entry_file_folder = 'svy_period_entry_datasets'
pandas_analyses_file_folder = 'pandas_period_datasets'

In [4]:
df = pd.read_stata(
'/home/kjb3/D1V1/kjb3largefiles/large_programming/datasets/\
cps/cps_employment_trends.dta',
convert_categoricals=True)

print("The DataFrame is", len(df), "rows long.")

# Original name: cps_00022.dta. (Giving these files a standard name allows
# me to overwrite older copies of the data that are no longer needed,
# thus preventing giant CPS files from piling up on my hard drive!)

# Converting all categorical dtypes to their underlying value:
# NOTE: This appears to dramatically reduce the amount of RAM *and*
# time needed to complete svy-based analyses!

for c in df.columns:
    if str(df[c].dtype) == 'category':
        df[c] = df[c].astype(df[c].dtype.categories.dtype)

# Note: This dataset was generated via https://cps.ipums.org/. 

# df

The DataFrame is 23488604 rows long.


Capitalizing the first letter of non-missing occupation responses so that they'll display better within our dashboard:

In [5]:
df['occ2010'] = np.where(df['occ2010'] != 'niu', df['occ2010'].str.capitalize(), df['occ2010'])

In [6]:
df['employed'] = np.where(df['empstat'].isin(['at work', 'has job, not at work last week']),
                          1, 0)

In [7]:
df['employed'].value_counts()

employed
0    12431937
1    11056667
Name: count, dtype: int64

In [8]:
df['empstat'].value_counts()

empstat
at work                           10641982
niu                                4317373
nilf, retired                      3737487
nilf, other                        2666344
nilf, unable to work                994370
unemployed, experienced worker      579011
has job, not at work last week      414685
armed forces                         79028
unemployed, new worker               58324
Name: count, dtype: int64

In [9]:
df['age_as_int'] = df['age'].astype('str').replace('under 1 year', '0').astype('int')

In [10]:
df['age_range'] = np.select(condlist = [df['age_as_int'] <= 21,
                      df['age_as_int'] <= 24,
                      df['age_as_int'] <= 29,
                      df['age_as_int'] <= 34,
                      df['age_as_int'] <= 39,
                      df['age_as_int'] <= 44,
                      df['age_as_int'] <= 49,
                      df['age_as_int'] <= 54,
                      df['age_as_int'] <= 59,
                      df['age_as_int'] >= 60],

          choicelist = ['Under 22',
                        '22-24',
                        '25-29',
                        '30-34',
                        '35-39',
                        '40-44',
                        '45-49',
                        '50-54',
                        '55-59',
                        '60 or older'],
          default = 'Missing age')
          

In [11]:
df['age_range'].value_counts()

age_range
Under 22       6337140
60 or older    5681832
55-59          1593975
50-54          1588313
30-34          1520942
45-49          1504760
35-39          1502961
40-44          1495920
25-29          1444533
22-24           818228
Name: count, dtype: int64

In [12]:
df['month_code'] = df['month'].map({'january':'01', 'february':'02', 
'march':'03', 'april':'04', 'may':'05', 'june':'06', 'july':'07', 'august':'08',
'september':'09', 'october':'10', 'november':'11', 'december':'12'})
                                      

In [13]:
df['quarter'] = df['month'].map({'january':'Q1', 'february':'Q1', 
'march':'Q1', 'april':'Q2', 'may':'Q2', 'june':'Q2', 'july':'Q3', 'august':'Q3',
'september':'Q3', 'october':'Q4', 'november':'Q4', 'december':'Q4'})
                                      

In [14]:
df['half'] = df['quarter'].map({'Q1':'H1', 'Q2':'H1', 'Q3':'H2', 'Q4':'H2'})
                                      

In [15]:
df['year_and_month'] = df['year'].astype('str')+'-'+df['month_code'].astype('str')

## Creating an initial set of strata values:

(This approach was suggested by Ivan Strahof of IPUMS at https://forum.ipums.org/t/calculating-standard-errors-using-cps-basic-monthly-microdata/6408/7 . It's based on guidance from Davern et al. (2007). I'll soon create an updated version of these strata such that every stratum is found within all year-month pairs, which Ivan also recommended.)

In [16]:
df['stratum'] = df['placefips'].astype('str') + '_' + df['county'].astype(
'str') + '_' + df['statefip'].astype('str')

In [17]:
df.query("placefips != 0")['stratum'].value_counts()

stratum
50000_11001_district of columbia    230080
44000_6037_california               201683
14000_17031_illinois                116451
35000_48201_texas                   105794
55000_4013_arizona                   78945
                                     ...  
81666_6095_california                 3295
82590_6071_california                 3208
64000_51740_virginia                  2721
35000_51650_virginia                  2635
49000_26163_michigan                  1615
Name: count, Length: 106, dtype: int64

In [18]:
df['stratum'].value_counts()

stratum
0_0_texas                 836028
0_0_illinois              430336
0_0_ohio                  420335
0_0_west virginia         380496
0_0_georgia               365764
                           ...  
0_37019_north carolina       193
0_1079_alabama               161
0_37141_north carolina       156
0_53015_washington            93
0_8014_colorado               24
Name: count, Length: 530, dtype: int64

In [19]:
# df.query("year == 2026")['cpsidp'].unique()

## Updating strata values

Ivan Strahof at IPUMS recommended that "it may make sense to restrict strata to only COUNTY and PLACEFIPS codes that appear across all vintages." Therefore, if certain strata don't appear for all year/month pairs, we'll replace their 'placefips' component with 0 in order to merge them into more general strata. If those county-statefips codes, in turn, don't appear across all strata, we'll merge them into the general statefips codes that begin with 0_0_.

### Determining which strata appear within all year/month pairs:

In [20]:
df_unique_year_month_stratum_groups = df[[
'year_and_month', 'stratum']].drop_duplicates().copy()
df_unique_year_month_stratum_groups

,year_and_month,stratum
0,2010-01,0_1003_alabama
3,2010-01,0_0_alabama
25,2010-01,0_1073_alabama
63,2010-01,0_1097_alabama
83,2010-01,0_1117_alabama
...,...,...
23436101,2026-08,0_39109_ohio
23443264,2026-08,0_26025_michigan
23450393,2026-08,0_13113_georgia
23459369,2026-08,0_48181_texas


In [21]:
df_months_with_stratum_data = df_unique_year_month_stratum_groups.pivot_table(
index = 'stratum', values = 'year_and_month', 
aggfunc = 'count').reset_index().sort_values('year_and_month', ascending = False)
df_months_with_stratum_data.rename(
columns = {'year_and_month':'year_month_pairs_with_stratum_data'}, inplace = True)
df_months_with_stratum_data.to_csv(
'Datasets/stratum_for_each_month.csv', index = False)

In [22]:
df_months_with_stratum_data['year_month_pairs_with_stratum_data'].value_counts(normalize=True)

year_month_pairs_with_stratum_data
199    0.545283
147    0.135849
67     0.115094
144    0.041509
58     0.033962
146    0.022642
12     0.022642
70     0.020755
198    0.011321
66     0.007547
11     0.005660
197    0.003774
196    0.003774
143    0.003774
142    0.003774
195    0.001887
145    0.001887
141    0.001887
140    0.001887
139    0.001887
65     0.001887
64     0.001887
62     0.001887
15     0.001887
14     0.001887
4      0.001887
3      0.001887
Name: proportion, dtype: float64

### Determining the maximum number of year-month pairs in which strata appear: 

(My new strata will all need to appear this many times, which will require consolidating quite a few of them.)

In [23]:
year_month_stratum_max = df_months_with_stratum_data[
'year_month_pairs_with_stratum_data'].max()
print(year_month_stratum_max)
df_months_with_stratum_data

199


,stratum,year_month_pairs_with_stratum_data
24,0_0_missouri,199
529,8954_6037_california,199
0,0_0_alabama,199
1,0_0_alaska,199
2,0_0_arizona,199
...,...,...
253,0_37129_north carolina,11
323,0_45085_south carolina,11
408,0_6101_california,11
414,0_8014_colorado,4


### Creating an initial set of revised strata:

This code checks whether a given strata appears within all year-month pairs. If it doesn't, the initial (placefips) component will be replaced with a 0, thus consolidating it into other strata that share the same county and state data, but don't have a specific placefips entry.

In [24]:
df_months_with_stratum_data['first_stratum_revision'] = np.where(
(df_months_with_stratum_data[
'year_month_pairs_with_stratum_data'] != year_month_stratum_max) & 
(df_months_with_stratum_data['stratum'].str.split('_').str[0] != '0'), 
'0_'+ (df_months_with_stratum_data['stratum'].str.split(
'_').str[1:]).str.join('_'),
df_months_with_stratum_data[
'stratum'])
         
df_months_with_stratum_data['first_stratum_revision']

24               0_0_missouri
529      8954_6037_california
0                 0_0_alabama
1                  0_0_alaska
2                 0_0_arizona
                ...          
253    0_37129_north carolina
323    0_45085_south carolina
408         0_6101_california
414           0_8014_colorado
364        0_53015_washington
Name: first_stratum_revision, Length: 530, dtype: str

In [25]:
df_months_with_stratum_data

,stratum,year_month_pairs_with_stratum_data,first_stratum_revision
24,0_0_missouri,199,0_0_missouri
529,8954_6037_california,199,8954_6037_california
0,0_0_alabama,199,0_0_alabama
1,0_0_alaska,199,0_0_alaska
2,0_0_arizona,199,0_0_arizona
...,...,...,...
253,0_37129_north carolina,11,0_37129_north carolina
323,0_45085_south carolina,11,0_45085_south carolina
408,0_6101_california,11,0_6101_california
414,0_8014_colorado,4,0_8014_colorado


## Merging these revised strata values into our list of all unique year-month/strata pairs:

In [26]:
df_unique_year_month_stratum_groups_first_revision = (
df_unique_year_month_stratum_groups.merge(df_months_with_stratum_data[
['stratum', 'first_stratum_revision']], on = 'stratum', how = 'left'))
df_unique_year_month_stratum_groups_first_revision.drop_duplicates(['year_and_month', 'first_stratum_revision'], inplace = True)
df_unique_year_month_stratum_groups_first_revision

,year_and_month,stratum,first_stratum_revision
0,2010-01,0_1003_alabama,0_1003_alabama
1,2010-01,0_0_alabama,0_0_alabama
2,2010-01,0_1073_alabama,0_1073_alabama
3,2010-01,0_1097_alabama,0_1097_alabama
4,2010-01,0_1117_alabama,0_1117_alabama
...,...,...,...
82883,2026-08,0_39109_ohio,0_39109_ohio
82884,2026-08,0_26025_michigan,0_26025_michigan
82885,2026-08,0_13113_georgia,0_13113_georgia
82886,2026-08,0_48181_texas,0_48181_texas


### Confirming that our np.where() code that was meant to replace 'placefips' values for strata that weren't present in all year/month pairs with 0s worked as intended:

In [27]:
df_unique_year_month_stratum_groups_first_revision.query("stratum != first_stratum_revision").drop_duplicates('first_stratum_revision').head(10)

,year_and_month,stratum,first_stratum_revision
7,2010-01,37000_9003_connecticut,0_9003_connecticut
165,2010-01,64000_51740_virginia,0_51740_virginia
200,2010-01,53980_6059_california,0_6059_california
263,2010-01,53775_20091_kansas,0_20091_kansas
326,2010-01,1000_51510_virginia,0_51510_virginia
384,2010-01,69084_6085_california,0_6085_california
394,2010-01,35000_51650_virginia,0_51650_virginia
437,2010-02,80000_6037_california,0_6037_california
852,2010-03,33000_6001_california,0_6001_california
4437,2010-12,49000_26163_michigan,0_26163_michigan


In [28]:
### Creating a new set of year/month counts for our revised strata:

In [29]:
df_months_with_stratum_data_first_revision = df_unique_year_month_stratum_groups_first_revision.pivot_table(
index = 'first_stratum_revision', values = 'year_and_month', 
aggfunc = 'count').reset_index().sort_values('year_and_month', ascending = False)
df_months_with_stratum_data_first_revision.rename(
columns = {'year_and_month':'year_month_pairs_with_stratum_data'}, 
inplace = True)

df_months_with_stratum_data_first_revision

,first_stratum_revision,year_month_pairs_with_stratum_data
24,0_0_missouri,199
497,8954_6037_california,199
0,0_0_alabama,199
1,0_0_alaska,199
2,0_0_arizona,199
...,...,...
326,0_45085_south carolina,11
417,0_6101_california,11
255,0_37129_north carolina,11
423,0_8014_colorado,4


The percentage of strata that appear within all year/month pairs has increased somewhat, but is still well below 100:

In [30]:
df_months_with_stratum_data_first_revision[
'year_month_pairs_with_stratum_data'].value_counts(normalize=True)

year_month_pairs_with_stratum_data
199    0.594378
147    0.108434
67     0.104418
144    0.044177
58     0.036145
12     0.024096
146    0.022088
70     0.022088
198    0.010040
66     0.008032
11     0.006024
143    0.004016
196    0.002008
194    0.002008
65     0.002008
64     0.002008
15     0.002008
14     0.002008
4      0.002008
3      0.002008
Name: proportion, dtype: float64

### Creating a second set of revised strata: 

(For this revision, any values with non-zero county codes that don't appear in all year-month pairs will have that code replaced with a 0, thus making them identical to the 'generic' strata for their states.)

In [31]:
df_months_with_stratum_data_first_revision['second_stratum_revision'] = np.where(
(df_months_with_stratum_data_first_revision[
'year_month_pairs_with_stratum_data'] != year_month_stratum_max) & 
(df_months_with_stratum_data_first_revision['first_stratum_revision'].str.split('_').str[1] != '0'), 
df_months_with_stratum_data_first_revision['first_stratum_revision'].str.split(
'_').str[0] + '_0_' + (df_months_with_stratum_data_first_revision['first_stratum_revision'].str.split(
'_').str[2]),
df_months_with_stratum_data_first_revision[
'first_stratum_revision'])
         
df_months_with_stratum_data_first_revision

,first_stratum_revision,year_month_pairs_with_stratum_data,second_stratum_revision
24,0_0_missouri,199,0_0_missouri
497,8954_6037_california,199,8954_6037_california
0,0_0_alabama,199,0_0_alabama
1,0_0_alaska,199,0_0_alaska
2,0_0_arizona,199,0_0_arizona
...,...,...,...
326,0_45085_south carolina,11,0_0_south carolina
417,0_6101_california,11,0_0_california
255,0_37129_north carolina,11,0_0_north carolina
423,0_8014_colorado,4,0_0_colorado


### Merging original stratum values into this dataset in order to help facilitate a merge of the second_stratum_revision field into our main DataFrame:

In [32]:
df_months_with_stratum_data_first_revision = (
df_months_with_stratum_data_first_revision.merge(
df_months_with_stratum_data.drop(
'year_month_pairs_with_stratum_data', axis = 1), 
on = 'first_stratum_revision', how = 'left'))
df_months_with_stratum_data_first_revision

,first_stratum_revision,year_month_pairs_with_stratum_data,second_stratum_revision,stratum
0,0_0_missouri,199,0_0_missouri,0_0_missouri
1,0_0_missouri,199,0_0_missouri,38000_0_missouri
2,8954_6037_california,199,8954_6037_california,8954_6037_california
3,0_0_alabama,199,0_0_alabama,0_0_alabama
4,0_0_alaska,199,0_0_alaska,0_0_alaska
...,...,...,...,...
525,0_45085_south carolina,11,0_0_south carolina,0_45085_south carolina
526,0_6101_california,11,0_0_california,0_6101_california
527,0_37129_north carolina,11,0_0_north carolina,0_37129_north carolina
528,0_8014_colorado,4,0_0_colorado,0_8014_colorado


### Merging this second set of strata values into our list of unique year/month/strata pairs:

In [33]:
df_unique_year_month_stratum_groups_second_revision = (
df_unique_year_month_stratum_groups.merge(df_months_with_stratum_data_first_revision[
['stratum', 'second_stratum_revision']], on = 'stratum', how = 'left'))
df_unique_year_month_stratum_groups_second_revision

,year_and_month,stratum,second_stratum_revision
0,2010-01,0_1003_alabama,0_1003_alabama
1,2010-01,0_0_alabama,0_0_alabama
2,2010-01,0_1073_alabama,0_0_alabama
3,2010-01,0_1097_alabama,0_1097_alabama
4,2010-01,0_1117_alabama,0_0_alabama
...,...,...,...
82883,2026-08,0_39109_ohio,0_0_ohio
82884,2026-08,0_26025_michigan,0_0_michigan
82885,2026-08,0_13113_georgia,0_0_georgia
82886,2026-08,0_48181_texas,0_0_texas


In [34]:
df_unique_year_month_stratum_groups_second_revision.drop_duplicates(
['year_and_month', 'second_stratum_revision'], inplace = True)
df_unique_year_month_stratum_groups_second_revision

,year_and_month,stratum,second_stratum_revision
0,2010-01,0_1003_alabama,0_1003_alabama
1,2010-01,0_0_alabama,0_0_alabama
3,2010-01,0_1097_alabama,0_1097_alabama
5,2010-01,8000_9001_connecticut,8000_9001_connecticut
6,2010-01,0_0_connecticut,0_0_connecticut
...,...,...,...
82875,2026-08,0_22103_louisiana,0_22103_louisiana
82879,2026-08,0_17163_illinois,0_17163_illinois
82880,2026-08,0_42089_pennsylvania,0_42089_pennsylvania
82881,2026-08,0_26115_michigan,0_26115_michigan


### Creating a new pivot table that checks whether all of these revised values now appear within all year/month pairs:

In [35]:
df_months_with_stratum_data_second_revision = (
df_unique_year_month_stratum_groups_second_revision.pivot_table(
index = 'second_stratum_revision', values = 'year_and_month', 
aggfunc = 'count').reset_index()).sort_values(
'year_and_month', ascending = False)
df_months_with_stratum_data_second_revision.rename(
columns = {'year_and_month':'year_month_pairs_with_stratum_data'}, 
inplace = True)

df_months_with_stratum_data_second_revision


,second_stratum_revision,year_month_pairs_with_stratum_data
296,8954_6037_california,199
0,0_0_alabama,199
1,0_0_alaska,199
2,0_0_arizona,199
3,0_0_arkansas,199
...,...,...
12,0_0_illinois,199
13,0_0_indiana,199
14,0_0_iowa,199
15,0_0_kansas,199


Almost all of them do at this point! However, at least one statewide stratum (e.g., Deleware) doesn't appear in all year-month pairs. Therefore, in order to get all strata to appear in each sample, I'll replace all other strata for those states with their statewide strata. (I'll double-check whether this is indeed the best approach, though, since it does result in a loss of precision.)

Creating a list of states whose statewide stratum doesn't appear within all year-month pairs:

In [36]:
partial_coverage_states = (
df_months_with_stratum_data_second_revision.query(
"year_month_pairs_with_stratum_data != @year_month_stratum_max")[
'second_stratum_revision'].str.split('_').str[-1].to_list())

In [37]:
df_months_with_stratum_data_second_revision[
'final_stratum_revision'] = np.where(
df_months_with_stratum_data_second_revision[
'second_stratum_revision'].str.split('_').str[-1].isin(
partial_coverage_states),
'0_0_'+df_months_with_stratum_data_second_revision[
'second_stratum_revision'].str.split('_').str[-1], 
df_months_with_stratum_data_second_revision[
'second_stratum_revision'])

Confirming that this update was successful:

In [38]:
df_months_with_stratum_data_second_revision.query(
"final_stratum_revision.str.split('_').str[-1].isin(@partial_coverage_states)")

,second_stratum_revision,year_month_pairs_with_stratum_data,final_stratum_revision
50,0_10001_delaware,199,0_0_delaware
51,0_10003_delaware,199,0_0_delaware
52,0_10005_delaware,199,0_0_delaware
7,0_0_delaware,15,0_0_delaware


Now that we have a set of strata that appear in all months with data, we can go ahead and merge these new strata into our main DataFrame. In order to facilitate this merge, though, we'll first need to create a table of the original strata that correspond to our final strata.

In [39]:
df_months_with_stratum_data_final_revision = df_months_with_stratum_data_second_revision.merge(
df_months_with_stratum_data_first_revision.drop('year_month_pairs_with_stratum_data', axis = 1), 
on = 'second_stratum_revision', how = 'left')
df_months_with_stratum_data_final_revision

,second_stratum_revision,year_month_pairs_with_stratum_data,final_stratum_revision,first_stratum_revision,stratum
0,8954_6037_california,199,8954_6037_california,8954_6037_california,8954_6037_california
1,0_0_alabama,199,0_0_alabama,0_0_alabama,0_0_alabama
2,0_0_alabama,199,0_0_alabama,0_1081_alabama,0_1081_alabama
3,0_0_alabama,199,0_0_alabama,0_1117_alabama,0_1117_alabama
4,0_0_alabama,199,0_0_alabama,0_1073_alabama,0_1073_alabama
...,...,...,...,...,...
525,0_0_kansas,199,0_0_kansas,0_0_kansas,0_0_kansas
526,0_0_kansas,199,0_0_kansas,0_20091_kansas,53775_20091_kansas
527,0_0_kansas,199,0_0_kansas,0_20091_kansas,0_20091_kansas
528,0_0_kansas,199,0_0_kansas,0_20045_kansas,0_20045_kansas


Confirming that all revised strata finally have entries for every year/month pair:

In [40]:
df_final_stratum_revision_pivot = df_unique_year_month_stratum_groups.merge(
df_months_with_stratum_data_final_revision[
['stratum', 'final_stratum_revision']], on = 'stratum', how = 'left').drop_duplicates(
['year_and_month', 'final_stratum_revision']).pivot_table(
index = 'final_stratum_revision', 
values = 'year_and_month', aggfunc = 'count').reset_index()
print("Smallest year_and_month count within the dataset:", 
df_final_stratum_revision_pivot['year_and_month'].min())
df_final_stratum_revision_pivot

Smallest year_and_month count within the dataset: 199


,final_stratum_revision,year_and_month
0,0_0_alabama,199
1,0_0_alaska,199
2,0_0_arizona,199
3,0_0_arkansas,199
4,0_0_california,199
...,...,...
289,78582_6111_california,199
290,8000_9001_connecticut,199
291,82000_51810_virginia,199
292,84000_26099_michigan,199


Merging these final data into our main DataFrame:

In [41]:
df = df.merge(df_months_with_stratum_data_final_revision[['stratum', 'final_stratum_revision']], 
on = 'stratum', how = 'left')
# df

In [42]:
df['year_and_quarter'] = df['year'].astype('str')+'-'+df['quarter']

In [43]:
df['year_and_half'] = df['year'].astype('str')+'-'+df['half']

In [44]:
df['uhrsworkt_revised'] = df['uhrsworkt'].astype('str').replace(
{'niu':-1, 'hours vary':-1}).astype('int')
df['FTE'] = np.where(df['uhrsworkt_revised'] >= 35, 1, 0)

In [45]:
df['FTE'].value_counts()

FTE
0    15064007
1     8424597
Name: count, dtype: int64

In [46]:
df['empstat_condensed'] = df['empstat'].map({'at work': 'Employed',
 'unemployed, experienced worker': 'Unemployed',
 'nilf, retired': 'NILF_NIU',
 'nilf, unable to work': 'NILF_NIU',
 'nilf, other': 'NILF_NIU',
 'niu': 'NILF_NIU',
 'has job, not at work last week': 'Employed',
 'armed forces': 'Employed',
 'unemployed, new worker': 'NILF_NIU'})
df['empstat_condensed'].value_counts()

empstat_condensed
NILF_NIU      11773898
Employed      11135695
Unemployed      579011
Name: count, dtype: int64

In [47]:
df['occ2010'].value_counts()

occ2010
niu                                                                                                    11726495
Miscellaneous managers, including funeral service managers and postmasters and mail superintendents      377214
Driver/sales workers and truck drivers                                                                   264991
Elementary and middle school teachers                                                                    250716
First-line supervisors of retail sales workers                                                           248199
                                                                                                         ...   
Podiatrists                                                                                                 398
Textile bleaching and dyeing, and cutting machine setters, operators, and tenders                           385
Brokerage clerks                                                                                

In [48]:
df['Count'] = 1

## Creating revised weights for quarterly, half-year, and annual analyses:

In order to create accurate estimates and confidence intervals, we'll need to divide our monthly weights by the number of months within a given sample. The divisor to use will generally be 12 for years, 3 for months, and 6 for half-year estimates; however, we'll also need to account for missing (e.g. Oct 2025) and partial-year data.

For more on CPS weight adjustments, see https://forum.ipums.org/t/journalist-question-averaging-multiple-months/6058 and https://forum.ipums.org/t/how-to-count-people-in-a-specific-occupation-with-wtfinl/6742/2.

In [49]:
# Filtering out our DataFrame to include only unique year/month combinations:
# (This will prove useful for determining the number of months (i.e. samples)
# that appear within each quarter, half-year period, and year.
df_unique_years_and_months = df.drop_duplicates('year_and_month').copy().reset_index()[['year', 'year_and_month', 'year_and_quarter', 'year_and_half', 'Count']]
df_unique_years_and_months

,year,year_and_month,year_and_quarter,year_and_half,Count
0,2010,2010-01,2010-Q1,2010-H1,1
1,2010,2010-02,2010-Q1,2010-H1,1
2,2010,2010-03,2010-Q1,2010-H1,1
3,2010,2010-04,2010-Q2,2010-H1,1
4,2010,2010-05,2010-Q2,2010-H1,1
...,...,...,...,...,...
194,2026,2026-04,2026-Q2,2026-H1,1
195,2026,2026-05,2026-Q2,2026-H1,1
196,2026,2026-06,2026-Q2,2026-H1,1
197,2026,2026-07,2026-Q3,2026-H2,1


In [50]:
### Calculating the number of samples for each year, half-year, and quarter:

In [51]:
for c in ['year', 'year_and_half', 'year_and_quarter']:
    df_unique_years_and_months[f"samples_in_{c}"] = df_unique_years_and_months.groupby(c)['Count'].transform('sum')

# We could also have gotten these counts via code like the following:
df_unique_years_and_months.pivot_table(index = 'year', 
values = 'Count', aggfunc = 'sum')
    

df_unique_years_and_months

,year,year_and_month,year_and_quarter,year_and_half,Count,samples_in_year,samples_in_year_and_half,samples_in_year_and_quarter
0,2010,2010-01,2010-Q1,2010-H1,1,12,6,3
1,2010,2010-02,2010-Q1,2010-H1,1,12,6,3
2,2010,2010-03,2010-Q1,2010-H1,1,12,6,3
3,2010,2010-04,2010-Q2,2010-H1,1,12,6,3
4,2010,2010-05,2010-Q2,2010-H1,1,12,6,3
...,...,...,...,...,...,...,...,...
194,2026,2026-04,2026-Q2,2026-H1,1,8,6,3
195,2026,2026-05,2026-Q2,2026-H1,1,8,6,3
196,2026,2026-06,2026-Q2,2026-H1,1,8,6,3
197,2026,2026-07,2026-Q3,2026-H2,1,8,2,2


Merging this sample-count data back into our main DataFrame so that it can be used to create alternative weights:

In [52]:
df = df.merge(df_unique_years_and_months[['year_and_month', 
'samples_in_year', 'samples_in_year_and_half', 
'samples_in_year_and_quarter']], on = 'year_and_month')

# df

### Creating revised weights for years, half-years, and quarters based on the number of monthly samples in those periods:

In [53]:
for c in ['year', 'year_and_half', 'year_and_quarter']:
    df[f"{c}_wtfinl"] = df['wtfinl'] / df[f"samples_in_{c}"]

# df

In [54]:
df['year_as_str'] = df['year'].astype('str')

In [55]:
cols_to_keep = ['age', 'year', 'occ2010', 'age_range', 'employed', 
'year_and_month', 'year_and_quarter', 'year_and_half', 'Count', 
'year_wtfinl', 'year_and_half_wtfinl', 'year_and_quarter_wtfinl', 'wtfinl', 
'year_as_str', 'empstat_condensed',
'cpsid', 'final_stratum_revision', 'age_as_int']
df_condensed = df[cols_to_keep].copy()
# df_condensed

In [56]:
unique_years = list(df['year'].unique())
unique_years

[np.int16(2010),
 np.int16(2011),
 np.int16(2012),
 np.int16(2013),
 np.int16(2014),
 np.int16(2015),
 np.int16(2016),
 np.int16(2017),
 np.int16(2018),
 np.int16(2019),
 np.int16(2020),
 np.int16(2021),
 np.int16(2022),
 np.int16(2023),
 np.int16(2024),
 np.int16(2025),
 np.int16(2026)]

## Creating estimates for each period, both for all ages together and by age range

**To do: Update the following script so that, if an entry for the given year is already present in Datasets, the loop will import that CSV file, then add it to df_list_Full. (If it's not present, the loop will then generate new stats.) This will prevent you from having to uncessarily process the same data over and over again for older years whenever you get new monthly data for the current year. (You'd just need to delete the current year's data, though, or add it to a 'years_to_redo' list that could be incorporated into your `if` statement.)**

## Debugging RAM issues by testing out individual total and proportion calculations:

In [57]:
latest_year_and_month = df['year_and_month'].max()
latest_year_and_month

'2026-08'

Checking estimated employment numbers for the most recent year and month:

In [58]:
df_latest_occupational_employment = df.query(
"year_and_month == @latest_year_and_month & occ2010 != 'niu'").pivot_table(
index = 'occ2010', values = 'wtfinl', aggfunc = 'sum').sort_values(
'wtfinl', ascending = False).reset_index()
df_latest_occupational_employment

,occ2010,wtfinl
0,"Miscellaneous managers, including funeral serv...",7.434234e+06
1,Registered nurses,3.830629e+06
2,Driver/sales workers and truck drivers,3.810241e+06
3,Elementary and middle school teachers,3.563329e+06
4,First-line supervisors of retail sales workers,2.946595e+06
...,...,...
448,Recreational therapists,3.038272e+03
449,"Buyers and purchasing agents, farm products",1.165714e+03
450,Procurement clerks,1.160712e+03
451,New accounts clerks,8.159162e+02


Creating a list of occupations with at least 1 million workers: (This list will also get incorporated into our HTML file. Note that it may change slightly each month as new data becomes available.)

In [59]:
large_occupations = df_latest_occupational_employment.query(
"wtfinl >= 1000000")['occ2010'].to_list().copy()
large_occupations.sort()
large_occupations

['Accountants and auditors',
 'Bookkeeping, accounting, and auditing clerks',
 'Carpenters',
 'Cashiers',
 'Chief executives and legislators',
 'Childcare workers',
 'Combined food preparation and serving workers, including fast food',
 'Computer occupations, all other',
 'Construction laborers',
 'Construction managers',
 'Cooks',
 'Counselors',
 'Couriers and messengers',
 'Customer service representatives',
 'Driver/sales workers and truck drivers',
 'Electricians',
 'Elementary and middle school teachers',
 'Farmers, ranchers, and other agricultural managers',
 'Financial managers',
 'First-line supervisors of non-retail sales workers',
 'First-line supervisors of office and administrative support workers',
 'First-line supervisors of retail sales workers',
 'Food preparation workers',
 'Food service managers',
 'General and operations managers',
 'Grounds maintenance workers',
 'Health practitioner support technologists and  technicians',
 'Janitors and building cleaners',
 'Labor

In [60]:
period_entry_files = os.listdir(svy_period_entry_file_folder)
period_entry_files[0:5]

['empstat_condensed_prop_job_for_year_and_month_2015-11.csv',
 'empstat_condensed_prop_job_by_age_range_for_year_and_month_2018-06.csv',
 'Count_total_job_for_year_and_month_2021-06.csv',
 'Count_total_job_for_year_and_quarter_2016-Q4.csv',
 'empstat_condensed_prop_job_for_year_and_month_2014-04.csv']

In [61]:
# Specifying the weight value to use for each period:
period_weight_map = {'year_as_str':"year_wtfinl",
                     'year_and_half':"year_and_half_wtfinl",
                     'year_and_quarter':"year_and_quarter_wtfinl",
                     'year_and_month':"wtfinl"}

## Testing out perform_svy_analysis() function:

In [62]:
period = 'year_and_month'
period_entry = '2026-08'
df_query_string = 'year_and_month == @period_entry'
y = 'Count'

Specifying the design for the survey: (This is based on Ivan Stranhof's guidance and on the svy documentation at https://svylab.com/docs/svy/tutorials/estimation.html.)

In [63]:
design = svy.Design(wgt=period_weight_map[period],
stratum = 'final_stratum_revision', psu = 'cpsid')

Creating a simpler design merely for testing purposes:

In [64]:
# simpler_design = svy.Design(wgt=period_weight_map[period])

In [65]:
frac_val = 1 # The % of responses that will be included in the
# test sample. (Smaller values will result in faster, but less accurate,
# calculations.)
period_entry_sample = svy.Sample(data=pl.DataFrame(
df_condensed.query(df_query_string).sample(frac=frac_val)), 
design=design)

In [66]:
sample_filter = [svy.col("occ2010") != "niu", svy.col(
"employed") == 1, svy.col('occ2010').isin(large_occupations)]

In [67]:
comparison_list = ['occ2010', 'age_range']
by_list = [period] + comparison_list.copy()
by_list_overall = [period, 'occ2010']
by_list, by_list_overall

(['year_and_month', 'occ2010', 'age_range'], ['year_and_month', 'occ2010'])

In [68]:
period_entry_sample.data

svy_row_index,age,year,occ2010,age_range,employed,year_and_month,year_and_quarter,year_and_half,Count,year_wtfinl,year_and_half_wtfinl,year_and_quarter_wtfinl,wtfinl,year_as_str,empstat_condensed,cpsid,final_stratum_revision,age_as_int
u32,str,i16,str,str,i64,str,str,str,i64,f64,f64,f64,f64,str,str,f64,str,i64
0,"""52""",2026,"""Credit counselors and loan off…","""50-54""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,775.165863,3100.66345,3100.66345,6201.3269,"""2026""","""Employed""",2.0261e13,"""4000_13121_georgia""",52
1,"""30""",2026,"""Receptionists and information …","""30-34""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,543.646575,2174.5863,2174.5863,4349.1726,"""2026""","""Employed""",2.0251e13,"""0_0_missouri""",30
2,"""8""",2026,"""niu""","""Under 22""",0,"""2026-08""","""2026-Q3""","""2026-H2""",1,256.017775,1024.0711,1024.0711,2048.1422,"""2026""","""NILF_NIU""",2.0261e13,"""0_19163_iowa""",8
3,"""25""",2026,"""Financial managers""","""25-29""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,687.876763,2751.50705,2751.50705,5503.0141,"""2026""","""Employed""",2.0251e13,"""0_36061_new york""",25
4,"""21""",2026,"""Food servers, nonrestaurant""","""Under 22""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,118.924887,475.69955,475.69955,951.3991,"""2026""","""Employed""",2.0251e13,"""0_15003_hawaii""",21
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
92050,"""35""",2026,"""niu""","""35-39""",0,"""2026-08""","""2026-Q3""","""2026-H2""",1,799.301225,3197.2049,3197.2049,6394.4098,"""2026""","""NILF_NIU""",2.0261e13,"""0_0_texas""",35
92051,"""44""",2026,"""First-line supervisors of non-…","""40-44""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,674.4313,2697.7252,2697.7252,5395.4504,"""2026""","""Employed""",2.0261e13,"""0_0_michigan""",44
92052,"""64""",2026,"""Receptionists and information …","""60 or older""",1,"""2026-08""","""2026-Q3""","""2026-H2""",1,102.978413,411.91365,411.91365,823.8273,"""2026""","""Employed""",2.0261e13,"""0_0_south dakota""",64


In [69]:
period_entry_sample.wrangling.filter_records(
        sample_filter).data.to_pandas()[y].count()

np.int64(24012)

In [70]:
# period_entry_sample.data

## Handling singletons: 

I went with the 'center' method, because according to the documentation for R's 'survey' library, the corresponding 'survey' method for treating lonely PSUs (adjust) is conservative.

In [71]:
period_entry_sample_adjusted_lonely_psus = period_entry_sample.singleton.center()

In [72]:
# df_svy_result = perform_svy_analysis(
#     sample = period_entry_sample, y = 'Count', 
#     analysis_type = 'total', by = by_list_overall, where = sample_filter, 
#     col_suffix = '_emp_total')
# df_svy_result

In [73]:
df_svy_result = perform_svy_analysis(
    sample = period_entry_sample_adjusted_lonely_psus, y = 'Count', 
    analysis_type = 'total', by = by_list_overall, where = sample_filter, 
    col_suffix = '_emp_total')
df_svy_result

pivot_index and pivot_values: ['year_and_month', 'occ2010'] Count


,year_and_month,occ2010,est_emp_total,se_emp_total,lci_emp_total,uci_emp_total,cv_emp_total,df_emp_total,error_upper_emp_total,error_lower_emp_total,est_as_pct_emp_total,se_as_pct_emp_total,lci_as_pct_emp_total,uci_as_pct_emp_total,error_upper_as_pct_emp_total,error_lower_as_pct_emp_total,Response_Count_emp_total
0,2026-08,Accountants and auditors,1.813022e+06,101407.638656,1.613454e+06,2.012591e+06,0.055933,297,199568.562221,199568.562221,1.813022e+08,1.014076e+07,1.613454e+08,2.012591e+08,1.995686e+07,1.995686e+07,455
1,2026-08,"Bookkeeping, accounting, and auditing clerks",1.298035e+06,78294.612083,1.143821e+06,1.452248e+06,0.060318,246,154213.309661,154213.309661,1.298035e+08,7.829461e+06,1.143821e+08,1.452248e+08,1.542133e+07,1.542133e+07,369
2,2026-08,Carpenters,1.010561e+06,73715.754541,8.650135e+05,1.156109e+06,0.072945,165,145547.749423,145547.749423,1.010561e+08,7.371575e+06,8.650135e+07,1.156109e+08,1.455477e+07,1.455477e+07,274
3,2026-08,Cashiers,2.495778e+06,125057.166299,2.249994e+06,2.741561e+06,0.050107,440,245783.617827,245783.617827,2.495778e+08,1.250572e+07,2.249994e+08,2.741561e+08,2.457836e+07,2.457836e+07,621
4,2026-08,Chief executives and legislators,1.732583e+06,92569.967548,1.550530e+06,1.914636e+06,0.053429,356,182052.726235,182052.726235,1.732583e+08,9.256997e+06,1.550530e+08,1.914636e+08,1.820527e+07,1.820527e+07,506
5,2026-08,Childcare workers,9.967201e+05,77465.638241,8.436635e+05,1.149777e+06,0.077721,151,153056.524670,153056.524670,9.967201e+07,7.746564e+06,8.436635e+07,1.149777e+08,1.530565e+07,1.530565e+07,260
6,2026-08,"Combined food preparation and serving workers,...",1.036053e+06,77400.974547,8.831239e+05,1.188981e+06,0.074708,151,152928.762211,152928.762211,1.036053e+08,7.740097e+06,8.831239e+07,1.188981e+08,1.529288e+07,1.529288e+07,261
7,2026-08,"Computer occupations, all other",1.226903e+06,82821.309164,1.063420e+06,1.390387e+06,0.067504,171,163483.794282,163483.794282,1.226903e+08,8.282131e+06,1.063420e+08,1.390387e+08,1.634838e+07,1.634838e+07,295
8,2026-08,Construction laborers,2.369008e+06,130157.806898,2.113107e+06,2.624909e+06,0.054942,389,255900.799480,255900.799480,2.369008e+08,1.301578e+07,2.113107e+08,2.624909e+08,2.559008e+07,2.559008e+07,583
9,2026-08,Construction managers,1.179303e+06,78339.175435,1.024892e+06,1.333714e+06,0.066428,215,154411.146362,154411.146362,1.179303e+08,7.833918e+06,1.024892e+08,1.333714e+08,1.544111e+07,1.544111e+07,343


In [74]:
# period_entry_sample.singleton.summary()

In [75]:
prop_sample_filter = [

    svy.col('empstat_condensed').isin(['Employed', 'Unemployed']),
    svy.col('occ2010').isin(large_occupations)
]

prop_sample_filter

[Expr(<Expr ['col("empstat_condensed").is_in…'] at 0x7A7E7FE8CC50>),
 Expr(<Expr ['col("occ2010").is_in([["Accoun…'] at 0x7A7E7FE8D450>)]

In [76]:
period_entry_sample.wrangling.filter_records(
prop_sample_filter).data.to_pandas().pivot_table(
index = ['year_and_half', 'occ2010', 'empstat_condensed'], 
values = 'Count',
aggfunc = 'count').reset_index()

,year_and_half,occ2010,empstat_condensed,Count
0,2026-H2,Accountants and auditors,Employed,455
1,2026-H2,Accountants and auditors,Unemployed,9
2,2026-H2,"Bookkeeping, accounting, and auditing clerks",Employed,369
3,2026-H2,"Bookkeeping, accounting, and auditing clerks",Unemployed,2
4,2026-H2,Carpenters,Employed,274
...,...,...,...,...
98,2026-H2,Stock clerks and order fillers,Unemployed,41
99,2026-H2,Teacher assistants,Employed,318
100,2026-H2,Teacher assistants,Unemployed,33
101,2026-H2,Waiters and waitresses,Employed,428


In [77]:
period_entry_sample_adjusted_lonely_psus

In [78]:
prop_sample_filter

[Expr(<Expr ['col("empstat_condensed").is_in…'] at 0x7A7E7FE8CC50>),
 Expr(<Expr ['col("occ2010").is_in([["Accoun…'] at 0x7A7E7FE8D450>)]

In [79]:
by_list_overall

['year_and_month', 'occ2010']

In [80]:
df_svy_result = perform_svy_analysis(
    sample = period_entry_sample_adjusted_lonely_psus, y = 'empstat_condensed', 
    analysis_type = 'prop', by = by_list_overall, where = prop_sample_filter, 
    col_suffix = '_emp_prop', count_col = 'Count')
df_svy_result

pivot_index and pivot_values: ['year_and_month', 'occ2010', 'empstat_condensed'] Count


,year_and_month,occ2010,empstat_condensed_emp_prop,est_emp_prop,se_emp_prop,lci_emp_prop,uci_emp_prop,cv_emp_prop,df_emp_prop,error_upper_emp_prop,error_lower_emp_prop,est_as_pct_emp_prop,se_as_pct_emp_prop,lci_as_pct_emp_prop,uci_as_pct_emp_prop,error_upper_as_pct_emp_prop,error_lower_as_pct_emp_prop,Response_Count_emp_prop
0,2026-08,Accountants and auditors,Employed,0.978986,0.007212,0.958967,0.989346,0.007367,304,0.010361,0.020018,97.898566,0.721190,95.896730,98.934629,1.036063,2.001836,455.0
1,2026-08,Accountants and auditors,NILF_NIU,0.000000,0.000000,0.000000,0.000000,inf,304,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,2026-08,Accountants and auditors,Unemployed,0.021014,0.007212,0.010654,0.041033,0.343190,304,0.020018,0.010361,2.101434,0.721190,1.065371,4.103270,2.001836,1.036063,9.0
3,2026-08,"Bookkeeping, accounting, and auditing clerks",Employed,0.987936,0.008587,0.951957,0.997054,0.008691,248,0.009118,0.035979,98.793605,0.858661,95.195670,99.705406,0.911800,3.597935,369.0
4,2026-08,"Bookkeeping, accounting, and auditing clerks",NILF_NIU,0.000000,0.000000,0.000000,0.000000,inf,248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,2026-08,Teacher assistants,NILF_NIU,0.000000,0.000000,0.000000,0.000000,inf,212,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
152,2026-08,Teacher assistants,Unemployed,0.103976,0.018899,0.072179,0.147551,0.181764,212,0.043575,0.031796,10.397561,1.889898,7.217941,14.755108,4.357547,3.179621,33.0
153,2026-08,Waiters and waitresses,Employed,0.959917,0.010874,0.932116,0.976618,0.011328,287,0.016701,0.027801,95.991656,1.087356,93.211557,97.661770,1.670114,2.780098,428.0
154,2026-08,Waiters and waitresses,NILF_NIU,0.000000,0.000000,0.000000,0.000000,inf,287,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN


In [81]:
df_svy_result.query("occ2010.str.contains('Software')")

,year_and_month,occ2010,empstat_condensed_emp_prop,est_emp_prop,se_emp_prop,lci_emp_prop,uci_emp_prop,cv_emp_prop,df_emp_prop,error_upper_emp_prop,error_lower_emp_prop,est_as_pct_emp_prop,se_as_pct_emp_prop,lci_as_pct_emp_prop,uci_as_pct_emp_prop,error_upper_as_pct_emp_prop,error_lower_as_pct_emp_prop,Response_Count_emp_prop
144,2026-08,"Software developers, applications and systems ...",Employed,0.979092,0.006277,0.962447,0.988447,0.006411,411,0.009356,0.016645,97.909163,0.627702,96.244693,98.844743,0.93558,1.66447,596.0
145,2026-08,"Software developers, applications and systems ...",NILF_NIU,0.000000,0.000000,0.000000,0.000000,inf,411,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,NaN
146,2026-08,"Software developers, applications and systems ...",Unemployed,0.020908,0.006277,0.011553,0.037553,0.300215,411,0.016645,0.009356,2.090837,0.627702,1.155257,3.755307,1.66447,0.93558,13.0


In [82]:
# df_svy_result.query("empstat_condensed_emp_prop == 'Unemployed'")

## Creating both overall estimates and estimates by age range for various periods:

In order to prevent my computer from running out of RAM, I ended up performing individual analyses for each period entry (e.g. 2025, 2023-01, 2024-Q3), then saving each one to a .csv file. All of these .csv files will then get combined into a single .csv file shortly.

In [83]:
if run_svy_analyses == True:
    for y, analysis_type in [
        ['Count', 'total'], ['empstat_condensed', 'prop']]:
        if y == 'Count':
            col_suffix = '_emp_total'
            count_col = '' # Won't be used for this analysis
        elif y == 'empstat_condensed':
            col_suffix = '_ue_prop'
            count_col = 'Count'
        else:
            raise ValueError("Unrecognized comparison-type value!")
        for comparison_type in ['job', 'job_by_age_range']:
            
            for period in ['year_as_str', 'year_and_half', 'year_and_quarter', 
        'year_and_month']:
                # Determining all of the period entries within this sample:
                # (To reduce our RAM usage, we'll perform individual analyses for each
                # period entry.)
                unique_periods = df[period].unique().copy()
                for period_entry in unique_periods:
                    df_query_string = f"{period} == @period_entry"
                    # Specifying which comparison variables to use for 
                    # our table of sums:
                    if comparison_type == 'job':
                        comparison_list = ['occ2010']
                        
                        # The default sample-filter argument will be used
                        # for our total-employment analyses. In this case,
                        # we'll want to filter out non-employed respondents
                        # and those who don't have an occupation listed.
                        sample_filter = [svy.col("occ2010") != "niu", 
                            svy.col("employed") == 1]
                        # Replacing this default argument with a custom one
                        # for our unemployment-proportion analyses:
                        # (Here, we do want to include both employed and
                        # unemployed respondents (but not those out of the
                        # labor force. We'll still want to remove
                        # 'niu' occupation values, though.)
                        if analysis_type == 'prop':
                            sample_filter = [svy.col("occ2010") != "niu", 
                            svy.col('empstat_condensed').isin(
                            ['Employed', 'Unemployed'])]
                    elif comparison_type == 'job_by_age_range':
                        comparison_list = ['occ2010', 'age_range']
                        # Because age-range-based data will have wider
                        # confidence intervals (and will consume more 
                        # memory), we'll limit our analyses to those
                        # occupations in our large_occupations list.
                        # Since 'niu' isn't in our large-occupations list,
                        # we don't have to filter for that variable in this
                        # case.
                        # Default filter for employment-total calculations:
                        sample_filter = [svy.col("employed") == 1, 
                            svy.col('occ2010').isin(large_occupations)]
                        # Changing the filter value for unemployment-%
                        # calculations:
                        if analysis_type == 'prop':
                            sample_filter = [
                            svy.col('empstat_condensed').isin(
                            ['Employed', 'Unemployed']),
                            svy.col('occ2010').isin(large_occupations)]
                    else:
                        raise ValueError(
                        "Unrecognized comparison-type value!")
                
                            
                    result_key = (y+'_'+analysis_type+'_'+comparison_type 
                    + '_for_' + period + '_' + period_entry)
                    filename = f'{result_key}.csv'
                    
                    by_list = [period] + comparison_list.copy()
            
            # To avoid out-of-memory errors (which I encountered even with 64 GB
            # of RAM plus a 70-GB swapfile), I'll perform analyses for one period 
            # entry (e.g. one year, one quarter of one year, etc.)
            # at a time. I don't believe that this will affect the accuracy of 
            # our confidence-interval calculations, as the CPS's weights were
            # originally calculated at the monthly level.
            
                    print("result_key:", result_key, "by_list:",
                    by_list, "sample_filter:", sample_filter,
                    "df_query_string:", df_query_string)
            
    
    
                    # Determining whether a new analysis needs to be 
                    # run for the active loop components:
                    
                    if filename in period_entry_files:
                        filename_contains_redo_substring = False
                        # Seeing whether a substring within a list of 
                        # filename substrings to redo is present within
                        # this filename: (If so, an analysis will still be
                        # run, and the existing file will get overwritten.)
                        for redo_substring in filename_substrings_to_redo:
                            if redo_substring in filename:
                                filename_contains_redo_substring = True
                                break
                        if filename_contains_redo_substring == False:
                            print(f"{filename} already present in folder. \
Skipping this analysis.")
                            continue

                    # Creating a sample design object for the active year
                    # and period type:
                    # (This code was based on the quickstart example at 
                    # https://svylab.com/docs/svy/#quick-start .)
            
                    # I would normally have made a copy of the DataFrame
                    # earlier in the loop in order to reduce the number of 
                    # query operations, but this approach should help
                    # save RAM while also preventing the original DataFrame
                    # from getting unintentionally modified.
            
                    # NOTE: It's important to only filter the DataFrame
                    # by the year at this point, as we'll need data for
                    # all respondents for each year (including those without
                    # jobs) in order to calculate accurate confidence intervals.

                    # Also note that I'm using the center() method, as
                    # discussed earlier, to deal with any lonely PSUs
                    # that may be present.

                    # Interestingly, specifying PSUs and strata
                    # appeared to speed up the analysis somewhat--perhaps
                    # because it allowed svy to perform calculations
                    # in smaller chunks? (For instance, per-year calculations
                    # initially took around 3 minutes, but now they
                    # take closer to 1 minute.)
                    
                    period_entry_sample = svy.Sample(data=pl.DataFrame(
                    df_condensed.query(df_query_string).copy()), 
                    # You could probably exclude the .copy() call here
                    # if needed if doing so would save RAM.
                    design=svy.Design(stratum='final_stratum_revision', psu='cpsid', 
                    wgt=period_weight_map[period])).singleton.center()
            
                    # When debugging this script, it can be helpful to add
                    # .sample(frac=0.001) to the output of .query(), 
                    # as doing so will
                    # dramatically reduce the time needed for the loop to
                    # finish running.
                    
                    # Passing additional filters to the sample *here*
                    # (within the estimation.total() call), rather than
                    # the previous query() call,
                    # will allow us to calculate proportions for our population
                    # of interest without distorting confidence intervals.
        
                    analysis_start_time = time.time()
                    # Performing our svy analysis:
                    # This will be either a total- or proportions-type
                    # estimate.
                    df_results_period_entry = perform_svy_analysis(
                        sample = period_entry_sample, y = y, 
                        analysis_type = analysis_type, by = by_list, where = sample_filter, 
                        col_suffix = col_suffix, count_col = count_col)
                    analysis_end_time = time.time()
                    print(f"Finished svy function in {round(
                    analysis_end_time - analysis_start_time, 3)} seconds.")
                    
                    # It's important to note that not everyone with an
                    # occ2010 entry is actually employed, as this value
                    # can also represent the last occupation in which someone
                    # worked before becoming unemployed.
                    # Also note that, in order to obtain accurate confidence
                    # intervals, the dataset being fed to estimation.total
                    # should include all CPS rows for a given year rather than
                    # just those who are employed. 
                    # See https://cps.ipums.org/cps-action/variables/OCC#description_section
                    # For more on OCC2010, see:
                    # https://cps.ipums.org/cps-action/variables/OCC2010#description_section
            
                    # Freeing up memory by deleting period_entry_sample
                    # and running a garbage collector:
                    # (Based on
                    # https://www.geeksforgeeks.org/python/how-to-explicitly-free-memory-in-python/
                    # )
                    # (Ended up removing this because it didn't solve
                    # the RAM-depletion issues I was facing)
                    # del period_entry_sample
                    # gc.collect()
                    
                    # Using standardized 'period' and 'period_val'
                    # fields will make it easier to concatenate
                    # tables for various results.
                    df_results_period_entry['period'] = period
                    df_results_period_entry.rename(
                    columns = {period:'period_val'}, inplace = True)
                    if comparison_type == 'job':
                        df_results_period_entry['age_range'] = 'All'
                        
                    print("Length of DataFrame:", len(df_results_period_entry), 
                    "\n")    
                    
                    df_results_period_entry.to_csv(svy_period_entry_file_folder+'/'+filename, index = False)

## Creating Pandas-based analyses for data-checking purposes:

These analyses will run more quickly but won't contain valuable confidence-interval data.

In [84]:
period = 'year_as_str'
comparison_list = ['occ2010']
by_list = [period] + comparison_list
weight_col = period_weight_map[period]
query_string = "occ2010 != 'niu' & employed == 1"
ones_col = 'Count'
col_suffix = '_emp_total_pd'
analysis_method = 'Total'
dv = ''
# Total-empolyment pivot:

# df_pivot = pd.pivot_table(df_condensed.query(query_string), 
# index=by_list, values = [weight_col, count_col], aggfunc = 'sum').reset_index()
# df_pivot.rename(columns = {count_col:'Response_Count',weight_col:'Total'}, inplace = True)
# if col_suffix != '':
#     col_renaming_dict = {c:c+col_suffix if c not in by_list else c 
#     for c in df_pivot.columns}
#     df_pivot.rename(columns = col_renaming_dict, inplace = True)

survey_pivot(df = df_condensed.query(query_string), 
weight_col = weight_col, analysis_method = analysis_method, 
by_list = by_list, dv = dv, col_suffix = col_suffix, ones_col = ones_col)

,year_as_str,occ2010,Response_Count_emp_total_pd,Total_emp_total_pd
0,2010,Accountants and auditors,8820,1.648526e+06
1,2010,Actors,124,2.494031e+04
2,2010,Actuaries,135,2.524293e+04
3,2010,Adhesive bonding machine operators and tenders,50,9.736348e+03
4,2010,Administrative services managers,615,1.044482e+05
...,...,...,...,...
7889,2026,"Welding, soldering, and brazing workers",1276,5.630762e+05
7890,2026,"Wholesale and retail buyers, except farm products",260,1.291595e+05
7891,2026,"Woodworking machine setters, operators, and te...",28,1.326961e+04
7892,2026,Word processors and typists,57,2.528534e+04


In [85]:
# Unemployment-proportion pivot:
period = 'year_as_str'
dv = 'empstat_condensed'
comparison_list = ['occ2010']
by_list = [period, 'age_range']#[period] + comparison_list
weight_col = period_weight_map[period]
query_string = "occ2010 != 'niu' & empstat_condensed in ['Employed', 'Unemployed']"
ones_col = 'Count'
col_suffix = '_ue_prop_pd'
analysis_method = 'Proportion'
comparison_type = 'job_by_age_range'

# df_pivot = pd.pivot_table(df_condensed.query(query_string), 
# index=by_list + [dv], values = [weight_col, count_col], aggfunc = 'sum').reset_index()
# df_pivot.rename(columns = {count_col:'Response_Count',weight_col:'Total'}, inplace = True)
# # Calculating proportions by dividing the 'Total' column for each row
# # by the sum of all 'Total' values for each set of by_list values:
# df_pivot['dv_prop'] = df_pivot['Total'] / df_pivot.groupby(
# by_list)['Total'].transform('sum')
# if col_suffix != '':
#     col_renaming_dict = {c:c+col_suffix if c not in by_list else c 
#     for c in df_pivot.columns}
#     df_pivot.rename(columns = col_renaming_dict, inplace = True)
                
df_results_for_period = survey_pivot(df = df_condensed.query(query_string), 
weight_col = weight_col, analysis_method = analysis_method, 
by_list = by_list, dv = dv, col_suffix = col_suffix, ones_col = ones_col)

df_results_for_period['period'] = period
df_results_for_period.rename(
columns = {period:'period_val'}, inplace = True)
if comparison_type == 'job':
    df_results_for_period['age_range'] = 'All'

df_results_for_period

,period_val,age_range,empstat_condensed_ue_prop_pd,Response_Count_ue_prop_pd,Total_ue_prop_pd,Proportion_ue_prop_pd,period
0,2010,22-24,Employed,39743,8.205933e+06,0.869508,year_as_str
1,2010,22-24,Unemployed,5566,1.231508e+06,0.130492,year_as_str
2,2010,25-29,Employed,77059,1.537788e+07,0.894338,year_as_str
3,2010,25-29,Unemployed,8357,1.816825e+06,0.105662,year_as_str
4,2010,30-34,Employed,77276,1.481249e+07,0.909145,year_as_str
...,...,...,...,...,...,...,...
335,2026,55-59,Unemployed,948,4.472541e+05,0.030203,year_as_str
336,2026,60 or older,Employed,60095,2.379093e+07,0.968713,year_as_str
337,2026,60 or older,Unemployed,1825,7.683814e+05,0.031287,year_as_str
338,2026,Under 22,Employed,21684,1.115024e+07,0.932902,year_as_str


In [86]:
ones_col = 'Count'

if run_pandas_analyses == True:
    for analysis_type in ['total_employment', 'unemployment_proportion']:
        if analysis_type == 'total_employment':
            col_suffix = '_emp_total_pd' # The 'pd' addition (which stands
            # for Pandas) will distinguish these columns from their
            # svy-based equivalents, which will prove useful when merging
            # svy- and Pandas-based analyses together.
            analysis_method = 'Total'
            dv = ''
        elif analysis_type == 'unemployment_proportion':
            col_suffix = '_ue_prop_pd'
            analysis_method = 'Proportion'
            dv = 'empstat_condensed'
        for comparison_type in ['job', 'job_by_age_range']:
            for period in ['year_as_str', 'year_and_half', 'year_and_quarter', 
        'year_and_month']:
                weight_col = period_weight_map[period]
                
                if comparison_type == 'job':
                    comparison_list = ['occ2010']
                    # Query string for total-employment calculations:
                    query_string = "occ2010 != 'niu' & employed == 1"
                    
                    if analysis_method == 'Proportion':
                        # Changing the query string for unemployment-%
                        # calculations:
                        query_string = "occ2010 != 'niu' & \
empstat_condensed in ['Employed', 'Unemployed']"
                        
                elif comparison_type == 'job_by_age_range':
                    comparison_list = ['occ2010', 'age_range']
                    # Query string for total-employment calculations:
                    query_string = "employed == 1 & occ2010 in \
@large_occupations"
                    
                    if analysis_method == 'Proportion':
                        # Changing the query string for unemployment-%
                        # calculations:
                        query_string = "empstat_condensed in \
['Employed', 'Unemployed'] & occ2010 in @large_occupations"
                        
                else:
                    raise ValueError(
                    "Unrecognized comparison-type value!")
            
                result_key = (analysis_type+'_'+comparison_type 
                + '_for_' + period)
                filename = f'{result_key}_pandas.csv'
                
                by_list = [period] + comparison_list.copy()
        
                print("result_key:", result_key, "by_list:",
                by_list, "query_string:", query_string, "weight_col",
                weight_col)
    
                analysis_start_time = time.time()
                # Performing our svy analysis:
                # This will be either a total- or proportions-type
                # estimate.
                df_results_for_period = survey_pivot(
                    df = df_condensed.query(query_string), 
                    weight_col = weight_col, 
                    analysis_method = analysis_method, 
                    by_list = by_list, dv = dv,
                    col_suffix = col_suffix, ones_col = ones_col)
                analysis_end_time = time.time()
                print(f"Finished Pandas-based survey_pivot() \
function in {round(analysis_end_time - analysis_start_time, 3)} seconds.")
                
                df_results_for_period['period'] = period
                df_results_for_period.rename(
                columns = {period:'period_val'}, inplace = True)
                if comparison_type == 'job':
                    df_results_for_period['age_range'] = 'All'
                    
                print("Length of DataFrame:", len(df_results_for_period), 
                "\n")    
                
                df_results_for_period.to_csv(
                pandas_analyses_file_folder+'/'+filename, index = False)

## Combining files created within the previous cells into new DataFrames:

## To do: consider updating the following code so that unemployment proportions are merged horizontally into the employment-totals table. (period, period_val, occ2010, and age_range should make for good merge keys.) 

(You could also remove the unique suffixes from both tables, then add a new table that clarifies what the values represent--but having the items side by side should work fine. Alternatively, you could keep these two datasets separate if the combined copy would be too large to upload to GitHub.)

In [87]:
# Importing all individual datasets created usisng the previous cell,
# together with any pre-existing datasets, into a single DataFrame:

df_full_emp_totals = pd.concat([pd.read_csv(
svy_period_entry_file_folder+'/'+filename) for 
filename in os.listdir(svy_period_entry_file_folder)
if ('Count_total' in filename)])

# Changing this field name to match the Pandas version for compatibility
# purposes:
df_full_emp_totals.rename(
columns = {'est_emp_total':'Total_emp_total'}, inplace = True)

# # Updating the capitalization of occupations in case they weren't
# # capitalized earlier:

# df_full_emp_totals['occ2010'] = df_full_emp_totals['occ2010'].str.capitalize()

df_full_emp_totals.sort_values(['period', 'period_val', 
'occ2010', 'age_range'], inplace = True)

# I hadn't included error-bar fields within all of my 
# total-employment .csv files, so I'll add in any such missing
# calculations here.
df_full_emp_totals['error_upper_emp_total'] = df_full_emp_totals[
'uci_emp_total'] - df_full_emp_totals['Total_emp_total']
df_full_emp_totals['error_lower_emp_total'] = df_full_emp_totals[
'Total_emp_total'] - df_full_emp_totals['lci_emp_total']

df_full_emp_totals.to_csv('Datasets/full_emp_totals.csv', index = False)

In [88]:
df_full_emp_totals

,period_val,occ2010,Total_emp_total,se_emp_total,lci_emp_total,uci_emp_total,cv_emp_total,df_emp_total,error_upper_emp_total,error_lower_emp_total,Response_Count_emp_total,period,age_range
0,2010-H1,Accountants and auditors,93416.485450,12113.519610,68742.053449,118090.917451,0.129672,32,24674.432001,24674.432001,216,year_and_half,22-24
1,2010-H1,Accountants and auditors,234778.698733,18694.349341,197832.202648,271725.194818,0.079625,146,36946.496085,36946.496085,566,year_and_half,25-29
2,2010-H1,Accountants and auditors,203440.958083,17644.756569,168493.294416,238388.621750,0.086732,116,34947.663667,34947.663667,555,year_and_half,30-34
3,2010-H1,Accountants and auditors,229360.245217,18394.195041,192996.181832,265724.308601,0.080198,141,36364.063385,36364.063385,579,year_and_half,35-39
4,2010-H1,Accountants and auditors,197474.858333,15542.813822,166741.997511,228207.719156,0.078708,138,30732.860822,30732.860822,554,year_and_half,40-44
...,...,...,...,...,...,...,...,...,...,...,...,...,...
451,2026,"Welding, soldering, and brazing workers",563076.186513,32469.594969,499251.705327,626900.667698,0.057665,417,63824.481185,63824.481185,1276,year_as_str,All
452,2026,"Wholesale and retail buyers, except farm products",129159.496137,14782.826977,99420.284201,158898.708074,0.114454,47,29739.211936,29739.211936,260,year_as_str,All
453,2026,"Woodworking machine setters, operators, and te...",13269.612000,4263.804033,1431.394162,25107.829838,0.321321,4,11838.217838,11838.217838,28,year_as_str,All
454,2026,Word processors and typists,25285.337275,5797.138760,-48374.294690,98944.969240,0.229269,1,73659.631965,73659.631965,57,year_as_str,All


## Modifying this dataset for use within our HTML site:

Converting population totals to integers not only makes the output more readable, but also reduces the size of the data file that we're about to create.

In [89]:
# Trimming out columns that won't be need for the dashboard should help
# our page load faster.
# NOTE: If you'll need to keep unemployment %s and employment totals separate in order to save space, you can create
# two separate cols-for-dashboard lists, one for each type, and then use those to more efficiently save your data to .txt/.csv files.
cols_for_dashboard = ['period', 'period_val', 'occ2010', 'Total_emp_total', 
'lci_emp_total', 'uci_emp_total', 'age_range', 
'Response_Count_emp_total']

In [90]:
df_full_emp_totals_for_site = df_full_emp_totals[cols_for_dashboard].copy()

for c in ['Total_emp_total', 'lci_emp_total', 'uci_emp_total']:
    df_full_emp_totals_for_site[c] = df_full_emp_totals_for_site[
    c].round(0).astype('Int64')
    # Casting to an Int64 in order to accommodate NaN values
df_full_emp_totals_for_site

,period,period_val,occ2010,Total_emp_total,lci_emp_total,uci_emp_total,age_range,Response_Count_emp_total
0,year_and_half,2010-H1,Accountants and auditors,93416,68742,118091,22-24,216
1,year_and_half,2010-H1,Accountants and auditors,234779,197832,271725,25-29,566
2,year_and_half,2010-H1,Accountants and auditors,203441,168493,238389,30-34,555
3,year_and_half,2010-H1,Accountants and auditors,229360,192996,265724,35-39,579
4,year_and_half,2010-H1,Accountants and auditors,197475,166742,228208,40-44,554
...,...,...,...,...,...,...,...,...
451,year_as_str,2026,"Welding, soldering, and brazing workers",563076,499252,626901,All,1276
452,year_as_str,2026,"Wholesale and retail buyers, except farm products",129159,99420,158899,All,260
453,year_as_str,2026,"Woodworking machine setters, operators, and te...",13270,1431,25108,All,28
454,year_as_str,2026,Word processors and typists,25285,-48374,98945,All,57


In [91]:
df_full_emp_totals_for_site.to_csv(
'Datasets/svy_emp_totals.csv', index = False)

Updating the csv_data.js file that will get used as a data source by the dashboard:

(To create the file, we'll simply add some JavaScript code to the front and end of a CSV render of df_full_emp_totals_for_site. This will be more efficient than manually copying and pasting the CSV data into our JavaScript file.)

In [92]:
csv_data_js_string = "let csv_data = `" + df_full_emp_totals_for_site.to_csv(
index=False) + "`;\nvar df_unfiltered = d3.csvParse(csv_data);"

In [93]:
with open ('csv_data.js', 'w') as file:
    file.write(csv_data_js_string)

## Creating option elements that can then get incorporated within our HTML page:

In [94]:
unique_age_ranges = list(df_full_emp_totals_for_site['age_range'].unique()).copy()
unique_age_ranges.remove('All')
unique_age_ranges.sort()
unique_age_ranges

['22-24',
 '25-29',
 '30-34',
 '35-39',
 '40-44',
 '45-49',
 '50-54',
 '55-59',
 '60 or older',
 'Under 22']

In [95]:
unique_occupations = [occ for occ in df_full_emp_totals_for_site[
'occ2010'].unique()].copy()
unique_occupations.sort()

Creating a string of JavaScript that will include occupation options in HTML format: (Large occupations will be flagged with an 'L' to make them easier to identify.)

In [96]:
all_occupations_string = "var full_occupation_options = `"

for occupation in unique_occupations:
    if occupation in large_occupations:
        large_occupation_prefix = "(L) "
    else:
        large_occupation_prefix = ""
#     print(f'<option value="{occupation}">{large_occupation_prefix}\
# {occupation}</option>')
    occupation_option_string =  f'<option value="{occupation}">{
large_occupation_prefix}{occupation}</option>\n'
    all_occupations_string += occupation_option_string

all_occupations_string += "`;\n\n" # The multiple line breaks will make
# it easier to distinguish between separate variables in our JavaScript
# file.

Creating a similar list for larger occupations:                       

In [97]:
large_occupations_string = "var large_occupation_options = `"
for occupation in large_occupations:
    large_occupations_string += f'<option value="{occupation}">(L) {occupation}</option>\n'

large_occupations_string += "`;\n\n"

In [98]:
large_occupations_array = "var large_occupations = " + str(large_occupations) + ";"

Combining these different variables into a single string:

In [99]:
occ_option_info_string = all_occupations_string + large_occupations_string + large_occupations_array

In [100]:
with open('occ_option_info.js', 'w') as file:
    file.write(occ_option_info_string)

# Specifying a discrete colorscale that can be used within JavaScript:

In [101]:
px.colors.qualitative.Dark24

['#2E91E5',
 '#E15F99',
 '#1CA71C',
 '#FB0D0D',
 '#DA16FF',
 '#222A2A',
 '#B68100',
 '#750D86',
 '#EB663B',
 '#511CFB',
 '#00A08B',
 '#FB00D1',
 '#FC0080',
 '#B2828D',
 '#6C7C32',
 '#778AAE',
 '#862A16',
 '#A777F1',
 '#620042',
 '#1616A7',
 '#DA60CA',
 '#6C4516',
 '#0D2A63',
 '#AF0038']

## Importing Pandas totals:

In [102]:
df_pd_emp_totals = pd.concat([pd.read_csv(
pandas_analyses_file_folder+'/'+filename) for 
filename in os.listdir(pandas_analyses_file_folder) if (
'total_employment' in filename)])

df_pd_emp_totals

,period_val,occ2010,Response_Count_emp_total_pd,Total_emp_total_pd,period,age_range
0,2010-H1,Accountants and auditors,4584,1.717245e+06,year_and_half,All
1,2010-H1,Actors,61,2.609359e+04,year_and_half,All
2,2010-H1,Actuaries,63,2.416238e+04,year_and_half,All
3,2010-H1,Adhesive bonding machine operators and tenders,24,9.914578e+03,year_and_half,All
4,2010-H1,Administrative services managers,303,1.024891e+05,year_and_half,All
...,...,...,...,...,...,...
92176,2026-08,"Weighers, measurers, checkers, and samplers, r...",25,9.339284e+04,year_and_month,All
92177,2026-08,"Welding, soldering, and brazing workers",132,4.575982e+05,year_and_month,All
92178,2026-08,"Wholesale and retail buyers, except farm products",33,1.074094e+05,year_and_month,All
92179,2026-08,Word processors and typists,7,2.945557e+04,year_and_month,All


In [103]:
df_condensed['employed_as_software_developer'] = np.where(
(df_condensed['occ2010'] == 'Software developers, applications \
and systems software') & (df_condensed['employed'] == 1), 1, 0)

#df

In [104]:
df_pd_emp_totals.to_csv(
'Datasets/pd_emp_totals.csv', index = False)

In [105]:
df_syv_pd_emp_comparison = df_full_emp_totals.merge(
df_pd_emp_totals, on = ['period', 'period_val', 'occ2010', 'age_range'], 
how = 'outer') # An outer merge will allow us to identify rows that
# are present in only one of the two datasets

df_syv_pd_emp_comparison['svy_pd_emp_total_diff'] = abs(
df_syv_pd_emp_comparison['Total_emp_total'] - df_syv_pd_emp_comparison['Total_emp_total_pd'])

df_syv_pd_emp_comparison.head()

,period_val,occ2010,Total_emp_total,se_emp_total,lci_emp_total,uci_emp_total,cv_emp_total,df_emp_total,error_upper_emp_total,error_lower_emp_total,Response_Count_emp_total,period,age_range,Response_Count_emp_total_pd,Total_emp_total_pd,svy_pd_emp_total_diff
0,2010-H1,Accountants and auditors,93416.485450,12113.519610,68742.053449,118090.917451,0.129672,32,24674.432001,24674.432001,216,year_and_half,22-24,216,93416.485450,7.275958e-11
1,2010-H1,Accountants and auditors,234778.698733,18694.349341,197832.202648,271725.194818,0.079625,146,36946.496085,36946.496085,566,year_and_half,25-29,566,234778.698733,5.820766e-11
2,2010-H1,Accountants and auditors,203440.958083,17644.756569,168493.294416,238388.621750,0.086732,116,34947.663667,34947.663667,555,year_and_half,30-34,555,203440.958083,0.000000e+00
3,2010-H1,Accountants and auditors,229360.245217,18394.195041,192996.181832,265724.308601,0.080198,141,36364.063385,36364.063385,579,year_and_half,35-39,579,229360.245217,2.910383e-11
4,2010-H1,Accountants and auditors,197474.858333,15542.813822,166741.997511,228207.719156,0.078708,138,30732.860822,30732.860822,554,year_and_half,40-44,554,197474.858333,2.037268e-10


Determining the maximum difference in proportions between these two datasets: (This value will most likely be greater than 0 due to minor rounding differences, or if one dataset is missing data for a certain time period.)

In [106]:
df_syv_pd_emp_comparison['svy_pd_emp_total_diff'].max()

np.float64(7.264316082000732e-08)

## Checking unemployment data:

**Note:** Due to low sample sizes, the ACS may be a better source of occupation-specific unemployment estimates than the CPS (though its data isn't as recent).

In [107]:
df_ue_props = pd.concat([pd.read_csv(
svy_period_entry_file_folder+'/'+filename, dtype={'period_val':'str'}) for 
filename in os.listdir(svy_period_entry_file_folder) if (
'prop_job' in filename)]).sort_values('period_val').reset_index(drop=True)

df_ue_props.to_csv('Datasets/full_ue_props.csv', index = False)

## Saving our revised microdata table data to a CSV file for use within other analyses

Note: Polars is **much** faster at exporting large CSV files than is Pandas. (The following export task took around 5 seconds with Polars, including the Pandas-to-Polars DataFrame conversion, but roughly 142 seconds within Pandas.

In [108]:
pl.DataFrame(df_condensed).write_csv('/home/kjb3/D1V1/kjb3largefiles/large_programming/\
datasets/cps/cps_employment_trends_revised.csv')

In [109]:
end_time = time.time()
print(f"{datetime.now().isoformat(sep = ' ', 
timespec = 'seconds')}: The script finished running in {
round(end_time - start_time, 3)} seconds.")

2026-09-22 22:21:08: The script finished running in 123.899 seconds.
